In [1]:
import re
import os
import time
import requests
import pandas as pd
from zipfile import ZipFile
from io import BytesIO
import warnings
warnings.filterwarnings('ignore')

# Configuration
MAX_RETRIES = 10
RETRY_DELAY = 30  # seconds
SURVEY_YEARS = list(range(2011, 2026))  # 2011 to 2025

print("Starting Stack Overflow Survey Data Download...")
print(f"Years to download: {min(SURVEY_YEARS)} to {max(SURVEY_YEARS)}")
print(f"Retry configuration: {MAX_RETRIES} max attempts, {RETRY_DELAY}s delay")
print(f"URL pattern: https://survey.stackoverflow.co/datasets/stack-overflow-developer-survey-{{year}}.zip\n")


Starting Stack Overflow Survey Data Download...
Years to download: 2011 to 2025
Retry configuration: 10 max attempts, 30s delay
URL pattern: https://survey.stackoverflow.co/datasets/stack-overflow-developer-survey-{year}.zip



In [2]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [3]:
def get_survey_urls(year):
    """
    Generate URL for a given survey year.
    All years use the same datasets ZIP pattern.
    """
    # All years use the same URL pattern
    url = f"https://survey.stackoverflow.co/datasets/stack-overflow-developer-survey-{year}.zip"
    return [url]

def fix_headers_for_older_years(df, year):
    """
    Fix headers for years 2011-2016 where the first two rows are headers.
    If the second row says "Response", use only the first row value.
    Otherwise, combine the first two rows.
    """
    if year >= 2011 and year <= 2015:
        if df.shape[0] < 2:
            print(f"  Warning: Not enough rows to fix headers for year {year}")
            return df
        
        # Get the first two rows
        first_row = df.iloc[0].astype(str)
        second_row = df.iloc[1].astype(str)
        
        # Create new column names
        new_columns = []
        for i, (first_val, second_val) in enumerate(zip(first_row, second_row)):
            first_val = first_val.strip()
            second_val = second_val.strip()
            
            # If second row is "Response", just use first row value
            if second_val == "Response":
                new_columns.append(first_val)
            else:
                # If both are the same or second is empty, use first
                if first_val == second_val or second_val == "":
                    new_columns.append(first_val)
                else:
                    # Combine both values - first value as primary
                    new_columns.append(f"{first_val} ({second_val})")
        
        # Set new column names
        df.columns = new_columns
        
        # Drop the first two rows (header rows)
        df = df.iloc[2:].reset_index(drop=True)
        
        print(f"  ✓ Fixed headers for year {year} (combined first two rows, removed 2 header rows)")
    
    return df

def download_file(url, year):
    """
    Download a file (single attempt, no retries).
    Returns the content if successful, None otherwise.
    Handles both CSV and ZIP files.
    """
    try:
        print(f"  Trying URL: {url}")
        response = requests.get(url, timeout=60, stream=True)
        response.raise_for_status()
        
        # Check content type
        content_type = response.headers.get('content-type', '').lower()
        
        # Reject HTML responses (likely error pages)
        if 'html' in content_type and response.status_code == 200:
            # Might be an error page, try next URL pattern
            print(f"  Warning: Received HTML instead of data file, may be wrong URL")
            return None
        
        content = response.content
        # Basic validation: check if content looks reasonable
        if len(content) < 100:
            print(f"  Warning: File too small, may be error page")
            return None
        
        # Check if it's a ZIP file by magic bytes
        is_zip = content[:2] == b'PK'  # ZIP files start with PK
        if is_zip:
            print(f"  ✓ Successfully downloaded {year} as ZIP ({len(content):,} bytes)")
        else:
            print(f"  ✓ Successfully downloaded {year} ({len(content):,} bytes)")
        
        return content
        
    except requests.exceptions.RequestException as e:
        print(f"  ✗ Download error: {str(e)}")
        return None


In [4]:
def validate_url(url):
    """
    Validate if a URL exists without downloading the full content.
    Returns True if the URL is valid and returns proper headers.
    """
    try:
        # Only get headers to check existence
        response = requests.head(url, timeout=10)
        return response.status_code == 200 and 'content-length' in response.headers
    except requests.exceptions.RequestException:
        return False

In [5]:
def download_and_extract_year(year, max_retries=MAX_RETRIES, delay=RETRY_DELAY, sample_size=None):
    """
    Download and extract survey data for a given year with retry logic.
    Tries multiple URL patterns and handles both CSV and ZIP files.
    Wraps the entire process in retry logic to catch any runtime errors.
    
    Args:
        year: The survey year to download
        max_retries: Maximum number of retry attempts
        delay: Delay between retries in seconds
        sample_size: If provided, only read this many rows from the CSV (for testing/development)
    
    Returns:
        DataFrame if successful, None otherwise.
    """
    print(f"\n{'='*60}")
    print(f"Processing year {year}")
    print(f"{'='*60}")
    
    urls = get_survey_urls(year)
    
    # Outer retry loop for entire download/extract process
    # This will retry the entire process up to max_retries times if a RuntimeError occurs
    for retry_attempt in range(max_retries):
        try:
            # Try each URL pattern
            for url in urls:
                content = download_file(url, year)
                
                if content is None:
                    continue
                
                # Check if content is a ZIP file by magic bytes (ZIP files start with 'PK')
                is_zip = content[:2] == b'PK'
                
                if is_zip:
                    # Try to parse as ZIP
                    try:
                        with ZipFile(BytesIO(content)) as zip_file:
                            # Look for CSV files in the ZIP (exclude macOS metadata)
                            csv_files = [f for f in zip_file.namelist() 
                                       if f.endswith('.csv') and not f.startswith('__MACOSX/')]
                            if csv_files:
                                # Use the first CSV file found
                                csv_file = csv_files[0]
                                print(f"  Found CSV file in ZIP: {csv_file}")
                                with zip_file.open(csv_file) as f:
                                    # For years 2011-2016, read without header to fix manually
                                    read_kwargs = {
                                        'low_memory': False, 
                                        'on_bad_lines': 'skip',
                                        'nrows': sample_size  # Add sample size parameter
                                    }
                                    if year >= 2011 and year <= 2015:
                                        read_kwargs['header'] = None
                                    
                                    try:
                                        df = pd.read_csv(f, encoding='utf-8', **read_kwargs)
                                        print(f"  ✓ Successfully loaded {year} from ZIP ({df.shape[0]:,} rows, {df.shape[1]:,} cols)")
                                        # Fix headers for older years
                                        df = fix_headers_for_older_years(df, year)
                                        return df
                                    except UnicodeDecodeError:
                                        f.seek(0)
                                        df = pd.read_csv(f, encoding='latin-1', **read_kwargs)
                                        print(f"  ✓ Successfully loaded {year} from ZIP with latin-1 encoding ({df.shape[0]:,} rows, {df.shape[1]:,} cols)")
                                        # Fix headers for older years
                                        df = fix_headers_for_older_years(df, year)
                                        return df
                            else:
                                print(f"  No CSV files found in ZIP archive")
                    except Exception as e:
                        print(f"  ZIP parsing failed: {str(e)}")
                        continue
                else:
                    # Try to parse as CSV directly
                    # For years 2011-2016, read without header to fix manually
                    read_kwargs = {
                        'low_memory': False, 
                        'on_bad_lines': 'skip',
                        'nrows': sample_size  # Add sample size parameter
                    }
                    if year >= 2011 and year <= 2016:
                        read_kwargs['header'] = None
                    
                    try:
                        df = pd.read_csv(BytesIO(content), encoding='utf-8', **read_kwargs)
                        print(f"  ✓ Successfully loaded {year} as CSV ({df.shape[0]:,} rows, {df.shape[1]:,} cols)")
                        # Fix headers for older years
                        df = fix_headers_for_older_years(df, year)
                        return df
                    except UnicodeDecodeError:
                        # Try different encoding
                        try:
                            df = pd.read_csv(BytesIO(content), encoding='latin-1', **read_kwargs)
                            print(f"  ✓ Successfully loaded {year} as CSV with latin-1 encoding ({df.shape[0]:,} rows, {df.shape[1]:,} cols)")
                            # Fix headers for older years
                            df = fix_headers_for_older_years(df, year)
                            return df
                        except Exception as e:
                            print(f"  CSV parsing failed: {str(e)}")
                            continue
                    except Exception as e:
                        print(f"  CSV parsing failed: {str(e)}")
                        continue
            
            # If we get here, the URL failed - this triggers a retry if attempts remain
            if retry_attempt < max_retries - 1:
                print(f"  ✗ Download failed for year {year}")
                print(f"  Retrying entire process (attempt {retry_attempt + 2}/{max_retries}) in {delay} seconds...")
                time.sleep(delay)
            else:
                print(f"  ✗ Failed to download and extract data for year {year} after {max_retries} attempts")
                print(f"  URL attempted: {urls[0]}")
                return None
                
        except RuntimeError as e:
            print(f"  ✗ Runtime error on attempt {retry_attempt + 1}: {str(e)}")
            if retry_attempt < max_retries - 1:
                print(f"  Retrying in {delay} seconds...")
                time.sleep(delay)
            else:
                print(f"  ✗ Failed after {max_retries} attempts due to runtime error")
                return None
        except Exception as e:
            # Catch any other unexpected errors and retry
            print(f"  ✗ Unexpected error on attempt {retry_attempt + 1}: {type(e).__name__}: {str(e)}")
            if retry_attempt < max_retries - 1:
                print(f"  Retrying in {delay} seconds...")
                time.sleep(delay)
            else:
                print(f"  ✗ Failed after {max_retries} attempts")
                return None
    
    return None

In [6]:
# Define sample size for testing (set to None for full dataset)
SAMPLE_SIZE = 1000  # Adjust this value to control how many rows to read from each year

# Download and create dataframes for each year
dataframes = {}

for year in SURVEY_YEARS:
    df = download_and_extract_year(year, max_retries=MAX_RETRIES, delay=RETRY_DELAY)
    if df is not None:
        # Add a year column to track which year the data is from
        df['SurveyYear'] = year
        dataframes[year] = df
    else:
        print(f"⚠ Skipping year {year} - download failed")

print(f"\n{'='*60}")
print(f"Download Summary")
print(f"{'='*60}")
print(f"Successfully downloaded: {len(dataframes)} out of {len(SURVEY_YEARS)} years")
print(f"Years downloaded: {sorted(dataframes.keys())}")
print(f"Years failed: {[y for y in SURVEY_YEARS if y not in dataframes]}")

# Display info for each dataframe
if dataframes:
    print(f"\n{'='*60}")
    print(f"DataFrame Information")
    print(f"{'='*60}")
    for year, df in sorted(dataframes.items()):
        print(f"Year {year}: {df.shape[0]:,} rows × {df.shape[1]:,} columns")


Processing year 2011
  Trying URL: https://survey.stackoverflow.co/datasets/stack-overflow-developer-survey-2011.zip
  ✓ Successfully downloaded 2011 as ZIP (80,173 bytes)
  Found CSV file in ZIP: 2011 Stack Overflow Survey Results.csv
  ✓ Successfully loaded 2011 from ZIP with latin-1 encoding (2,815 rows, 65 cols)
  ✓ Fixed headers for year 2011 (combined first two rows, removed 2 header rows)

Processing year 2012
  Trying URL: https://survey.stackoverflow.co/datasets/stack-overflow-developer-survey-2012.zip
  ✓ Successfully downloaded 2012 as ZIP (266,621 bytes)
  Found CSV file in ZIP: 2012 Stack Overflow Survey Results.csv
  ✓ Successfully loaded 2012 from ZIP with latin-1 encoding (6,245 rows, 75 cols)
  ✓ Fixed headers for year 2012 (combined first two rows, removed 2 header rows)

Processing year 2013
  Trying URL: https://survey.stackoverflow.co/datasets/stack-overflow-developer-survey-2013.zip
  ✓ Successfully downloaded 2013 as ZIP (689,493 bytes)
  Found CSV file in ZIP: 

### Create Combined Data Frame

In [7]:
# Create combined dataframe from all years
if dataframes:
    print(f"\n{'='*60}")
    print(f"Creating Combined DataFrame")
    print(f"{'='*60}")
    
    # Ensure all dataframes have unique columns before concatenation
    # Find the union of all columns
    all_columns = set()
    for df in dataframes.values():
        all_columns.update(df.columns)
    all_columns = list(all_columns)

    # Reindex each dataframe to ensure unique columns for concat
    aligned_dfs = []
    for year, df in dataframes.items():
        # Remove duplicate columns if any (can happen on bad CSVs)
        df = df.loc[:,~df.columns.duplicated()]
        aligned_df = df.reindex(columns=all_columns)
        aligned_dfs.append(aligned_df)
    
    combined_df = pd.concat(aligned_dfs, ignore_index=True, sort=False)
    
    print(f"success!")
    print(f"  rows: {combined_df.shape[0]:,}")
    print(f"  columns: {combined_df.shape[1]:,}")
    print(f"  Years: {sorted(combined_df['SurveyYear'].dropna().unique())}")
    
    # Show basic info about the combined dataframe
    print(f"\n{'='*60}")
    print(f"Combined DataFrame Info")
    print(f"{'='*60}")
    print(combined_df.info())
    print(f"{'='*60}")
else:
    print("\n⚠ No dataframes were successfully downloaded. Cannot create combined dataframe.")
    combined_df = None



Creating Combined DataFrame
success!
  rows: 772,667
  columns: 1,089
  Years: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Combined DataFrame Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 772667 entries, 0 to 772666
Columns: 1089 entries, SkipMeals to nan (PS4)
dtypes: float64(108), int64(1), object(980)
memory usage: 6.3+ GB
None


In [8]:
# Access individual year dataframes: dataframes[year]
# Access combined dataframe: combined_df
# Example:
if dataframes:
    print(f"\n{'='*60}")
    print(f"How to Access Your Data")
    print(f"{'='*60}")
    print(f"Individual year dataframes:")
    print(f"  - dataframes[2024]  # Access 2024 data")
    print(f"  - dataframes[2023]  # Access 2023 data")
    print(f"  - etc.")
    print(f"\nCombined dataframe:")
    print(f"  - combined_df  # All years combined")
    print(f"\nAvailable years: {sorted(dataframes.keys())}")
    
    # Quick preview of the combined dataframe
    if combined_df is not None:
        print(f"\n{'='*60}")
        print(f"Combined DataFrame Preview (first 5 rows)")
        print(f"{'='*60}")
        print(combined_df.head())



How to Access Your Data
Individual year dataframes:
  - dataframes[2024]  # Access 2024 data
  - dataframes[2023]  # Access 2023 data
  - etc.

Combined dataframe:
  - combined_df  # All years combined

Available years: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Combined DataFrame Preview (first 5 rows)
  SkipMeals nan (Most urgent info about job opportunity: Tech stack)  \
0       NaN                                                      NaN   
1       NaN                                                      NaN   
2       NaN                                                      NaN   
3       NaN                                                      NaN   
4       NaN                                                      NaN   

  nan (Open to new job opportunities) nan (Stack Overflow Careers Message)  \
0                                 NaN                                  NaN   
1                                 NaN                   

### Sample the dataframe (default - 20%)

In [16]:
# Create a stratified sample of 20% of the data
if combined_df is not None:
    # Calculate 20% sample size for each year
    sample_size = 0.2
    
    # Perform stratified sampling
    stratified_sample = combined_df.groupby('SurveyYear', group_keys=False).apply(
        lambda x: x.sample(frac=sample_size, random_state=42)
    ).reset_index(drop=True)
    
    # Export to CSV
    output_file = 'stackoverflow_survey_stratified_sample.csv'
    stratified_sample.to_csv(output_file, index=False)
    
    print(f"Original dataset size: {len(combined_df):,} rows")
    print(f"Sampled dataset size: {len(stratified_sample):,} rows")
    print(f"\nSample size by year:")
    print(stratified_sample['SurveyYear'].value_counts().sort_index())
    print(f"\nData exported to: {output_file}")

Original dataset size: 772,599 rows
Sampled dataset size: 154,521 rows

Sample size by year:
SurveyYear
2011      563
2012     1249
2013     1948
2014     1529
2015     5217
2016    11206
2017    10278
2018    19771
2019    17777
2020    12892
2021    16688
2022    14654
2023    17837
2024    13087
2025     9825
Name: count, dtype: int64

Data exported to: stackoverflow_survey_stratified_sample.csv


In [17]:
stratified_sample.to_csv('stackoverflow_survey_stratified_sample.csv')

### Adjust df_use based on the dataframe you want to use (sample_df, stratified_sample, etc)

In [18]:
sample_df = pd.read_csv('stackoverflow_survey_stratified_sample.csv')

KeyboardInterrupt: 

In [9]:
df_use = combined_df.copy()

### Adjust column names for Undefined/NaN

In [10]:
# Remove any columns with 'Unnamed' in their name from the combined dataframe
if df_use is not None:
    df_use = df_use.loc[:, ~df_use.columns.str.contains('^Unnamed')]

In [11]:
# Remove columns with 'nan' in their names from combined_df and individual year dataframes
if df_use is not None:
    # For combined dataframe
    nan_columns = df_use.columns[df_use.columns.str.contains('nan', case=False, na=False)]
    if len(nan_columns) > 0:
        print("Removing columns containing 'nan' from combined dataframe:")
        print(list(nan_columns))
        df_use = df_use.drop(columns=nan_columns)

print("\nDone cleaning column names.")

Removing columns containing 'nan' from combined dataframe:
['nan (Most urgent info about job opportunity: Tech stack)', 'nan (Open to new job opportunities)', 'nan (Stack Overflow Careers Message)', 'nan (Source control used: Legacy / Custom)', 'nan (Finance)', 'nan (Most annoying about job search: The Interview)', 'nan (Recommender)', 'nan (How frequently land on or read Stack Overflow)', 'nan (Why answer: Self promotion)', 'nan (Direct sales to consumers)', 'nan (Why answer: No idea)', 'nan (I click on ads that interest me)', 'nan (Current Lang & Tech: Spark)', 'nan (Stock Options/Profit Sharing Program)', 'nan (How often contacted by recruiters)', 'nan (JQuery)', 'nan (Future Lang & Tech: Haskell)', 'nan (Future Lang & Tech: Write-In)', 'nan (Source control used: CVS)', 'nan (Looking for a new job)', 'nan (No Involvement)', 'nan (PHP)', 'nan (How can companies improve interview process: Better preparation)', 'nan (Current Lang & Tech: MongoDB)', 'nan (Future Lang & Tech: CoffeeScrip

### Convert column names to lower case

In [12]:
# Update all column names in combined_df and each dataframe in dataframes to be lower case
df_use.columns = [col.lower() for col in df_use.columns]

### Identify and combine like columns

In [13]:
# Dictionary of column groups to combine
column_groups = {
    'years_coding': ['yearscode', 'yearscodingprof', 'yearscodepro', 'work_experience', 'yearsprogram', 'yearscodedjob', 'how many years of it/programming experience do you have?'],
    'education': ['edlevel', 'education', 'formaleducation'],
    'employment': ['employment', 'employmentstatus', 'employment_status'],
    'company_size': ['companysize', 'company_size_range', 'orgsize', 'companyemployeesrange', 'how many people work for your company?', 'which best describes the size of your company?'],
    'salary': ['convertedsalary', 'convertedcomp', 'comptotal'],
    'job_satisfaction': ['jobsatisfaction', 'job_satisfaction', 'careersatisfaction', 'please rate your job/career satisfaction', 'what best describes your career / job satisfaction?'],
    'job_title': ['jobtitle', 'currentjobtitle', 'jobprofile'],
    'developer_type': ['developertype', 'devtype'],
    'industry': ['industry', 'industrytype', 'companytype', 'how would you best describe the industry you work in?', 'how would you best describe the industry you currently work in?'],
    'country': ['country', 'location', 'countrycode', 'what country or region do you live in?', 'what country do you live in?'],
    'programming_experience': ['yearscode', 'yearscoding', 'codingexperience'],
    'database_worked_with': ['databaseworkedwith', 'dbworkedwith'],
    'dev_environment': ['ide', 'developmentenvironment', 'dev_environment', 'devenviron', 'devenvironment'],
    'operating_sys': ['opsys', 'operatingsystem', 'os', 'what operating system do you use the most?', 'which desktop operating system do you use the most?'],
    'dev_methodology': ['methodology', 'devmethodology', 'developmentmethodology'],
    'communication_tools': ['communicationtools', 'collaboration', 'collabtools'],
    'gender': ['gender', 'sex', 'what is your gender?'],
    'age': ['age', 'agerange','age_range','agegrouping', 'how old are you?'],
    'learning': ['learncode', 'learncodehow', 'learningmethod'],
    'work_experience': ['workexp', 'experience', 'yearsexperience'],
    'remote': ['remotework', 'workremote', 'remotestatus', 'do you work remotely?', 'homeremote', 'remote'],
    'team_size': ['teamsize', 'orgteamsize', 'developmentteamsize', 'how large is the team that you work on?'],
    'survey_easy': ['surveyease', 'surveyeasy', 'surveylong', 'surveytoolong', 'surveylength'],
    'version_control_sys': ['versioncontrol', 'versioncontrolsystem', 'vcs'],
    'currency': ['currency', 'currencydesc'],
    'hobby': ['hobby', 'hobbyist'],
    'race': ['race', 'raceethnicity', 'self_identification']
}

In [14]:
df_use_2 = df_use.copy()

In [15]:
# Identify and combine duplicate columns
def combine_duplicate_columns(df):
    """
    Identifies columns with the same name (case-insensitive), combines their data
    into a single column by taking the first non-null value, and removes the originals.

    Args:
        df: The input DataFrame.

    Returns:
        A new DataFrame with duplicate columns combined.
    """
    df_combined_duplicates = pd.DataFrame(index=df.index)
    processed_columns = set()

    for col_name in df.columns:
        col_name_lower = col_name.lower()

        if col_name_lower not in processed_columns:
            # Find all columns with this name (case-insensitive)
            duplicate_columns = [col for col in df.columns if col.lower() == col_name_lower]

            if len(duplicate_columns) > 1:
                print(f"Combining duplicate columns for '{col_name_lower}': {duplicate_columns}")
                # Select the duplicate columns
                selected_duplicates = df[duplicate_columns]
                # Combine by taking the first non-null value across rows
                combined_series = selected_duplicates.bfill(axis=1).iloc[:, 0]
                df_combined_duplicates[col_name_lower] = combined_series
                # Add to processed set
                processed_columns.add(col_name_lower)
            else:
                # Not a duplicate, just add the column
                df_combined_duplicates[col_name_lower] = df[col_name]
                processed_columns.add(col_name_lower)

    return df_combined_duplicates

# Apply the function to combine duplicate columns in df_use_2
df_use_combined_duplicates = combine_duplicate_columns(df_use_2)

print("\nOriginal DataFrame shape:", df_use_2.shape)
print("DataFrame shape after combining duplicates:", df_use_combined_duplicates.shape)

Combining duplicate columns for 'webframeworkedwith': ['webframeworkedwith', 'webframeworkedwith']
Combining duplicate columns for 'webframedesirenextyear': ['webframedesirenextyear', 'webframedesirenextyear']
Combining duplicate columns for 'gender': ['gender', 'gender']
Combining duplicate columns for 'hobby': ['hobby', 'hobby']
Combining duplicate columns for 'country': ['country', 'country']
Combining duplicate columns for 'industry': ['industry', 'industry']

Original DataFrame shape: (772667, 731)
DataFrame shape after combining duplicates: (772667, 725)


In [16]:
def combine_columns(df, column_list):
    """
    Combines data from a list of columns into a single Series,
    taking the first non-null value across the columns for each row.

    Args:
        df: The input pandas DataFrame.
        column_list: A list of column names to combine.

    Returns:
        A pandas Series containing the combined data.
    """
    # Select the specified columns
    selected_columns = df[column_list]

    # Combine columns by taking the first non-null value across rows
    combined_series = selected_columns.bfill(axis=1).iloc[:, 0]

    return combined_series

In [17]:
# Initialize an empty dictionary to store consolidated columns
consolidated_columns_dict = {}

# Iterate through the column_groups dictionary
for group_name, column_list in column_groups.items():
    # Identify columns in df_use_combined_duplicates that are present in the current group's list
    present_columns = [col for col in column_list if col in df_use_combined_duplicates.columns]

    # If there are columns from the current group present in the DataFrame
    if present_columns:
        print(f"Processing group '{group_name}' with columns: {present_columns}")
        # Call the combine_columns function
        combined_series = combine_columns(df_use_combined_duplicates, present_columns)
        # Store the resulting combined Series in the dictionary
        consolidated_columns_dict[group_name] = combined_series
    else:
        print(f"No columns found for group '{group_name}' in the DataFrame.")

# Create a new DataFrame from the dictionary of consolidated columns
df_consolidated = pd.DataFrame(consolidated_columns_dict)

print("\nConsolidated DataFrame created.")
print(f"Shape of consolidated DataFrame: {df_consolidated.shape}")
df_consolidated.head()

Processing group 'years_coding' with columns: ['yearscode', 'yearscodingprof', 'yearscodepro', 'yearsprogram', 'yearscodedjob', 'how many years of it/programming experience do you have?']
Processing group 'education' with columns: ['edlevel', 'education', 'formaleducation']
Processing group 'employment' with columns: ['employment', 'employmentstatus', 'employment_status']
Processing group 'company_size' with columns: ['companysize', 'company_size_range', 'orgsize', 'how many people work for your company?', 'which best describes the size of your company?']
Processing group 'salary' with columns: ['convertedsalary', 'convertedcomp', 'comptotal']
Processing group 'job_satisfaction' with columns: ['jobsatisfaction', 'job_satisfaction', 'careersatisfaction', 'please rate your job/career satisfaction', 'what best describes your career / job satisfaction?']
Processing group 'job_title' with columns: ['jobprofile']
Processing group 'developer_type' with columns: ['developertype', 'devtype']
Pr

,years_coding,education,employment,company_size,salary,job_satisfaction,job_title,developer_type,industry,country,programming_experience,database_worked_with,dev_environment,operating_sys,dev_methodology,communication_tools,gender,age,learning,work_experience,remote,team_size,survey_easy,version_control_sys,currency,hobby,race
0,<2,NaN,NaN,Start Up (1-25),NaN,FML,NaN,NaN,Consulting,Africa,NaN,NaN,NaN,Linux,NaN,NaN,NaN,< 20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,41310,NaN,NaN,Mature Small Business (25-100),NaN,So happy it hurts,NaN,NaN,Software Products,Other Europe,NaN,NaN,NaN,Windows 7,NaN,NaN,NaN,25-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,41435,NaN,NaN,Mid Sized (100-999),NaN,NaN,NaN,NaN,Software Products,India,NaN,NaN,NaN,Linux,NaN,NaN,NaN,25-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,41310,NaN,NaN,Student,NaN,I enjoy going to work,NaN,NaN,Foundation / Non-Profit,Germany,NaN,NaN,NaN,Linux,NaN,NaN,NaN,< 20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,11,NaN,NaN,Start Up (1-25),NaN,It pays the bills,NaN,NaN,Software Products,Other Asia,NaN,NaN,NaN,Linux,NaN,NaN,NaN,35-39,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# Create a list of all original column names that were combined
original_columns_to_drop = []
for group_name, column_list in column_groups.items():
    # Identify columns in df_use_combined_duplicates that are present in the current group's list
    present_columns = [col for col in column_list if col in df_use_combined_duplicates.columns]
    original_columns_to_drop.extend(present_columns)

# Remove duplicates from the list of columns to drop
original_columns_to_drop = list(set(original_columns_to_drop))

# Drop these original columns from the df_use_combined_duplicates DataFrame
df_use_combined_duplicates_dropped = df_use_combined_duplicates.drop(columns=original_columns_to_drop, errors='ignore')

print("Original columns that were combined have been dropped.")
print(f"Shape of df_use_combined_duplicates after dropping original columns: {df_use_combined_duplicates_dropped.shape}")

Original columns that were combined have been dropped.
Shape of df_use_combined_duplicates after dropping original columns: (772667, 652)


In [19]:
# Concatenate the remaining columns from df_use_combined_duplicates_dropped with the consolidated columns
df_all = pd.concat([df_use_combined_duplicates_dropped, df_consolidated], axis=1)

print("\nFinal DataFrame created by concatenating remaining original columns and consolidated columns.")
print(f"Shape of the final DataFrame: {df_all.shape}")

# Display the first few rows of the final dataframe
print("\nFirst 5 rows of the final DataFrame:")
display(df_all.head())


Final DataFrame created by concatenating remaining original columns and consolidated columns.
Shape of the final DataFrame: (772667, 679)

First 5 rows of the final DataFrame:


,skipmeals,employmentaddl,resumeupdate,"including yourself, how many developers are employed at your company?",professionalcloud,aimodelschoice,aiagentimpactsomewhat disagree,platformadmired,salary_midpoint,majorundergrad,newonboardgood,rep_range,techendorse_9,soai,how likely is it that a recommendation you make will be acted upon?,commplatformhaveworkedwith,which of the following best describes your occupation?,vchostingpersonal use,so_actions_10,"you answered you don't have a careers profile, can you elaborate why?",assessjobprofdevel,problemsolving,team_size_range,misctechdesirenextyear,aiagentobswrite,influencerecruitment,ergonomicdevices,assessjobcompensation,knowledge_3,have you visited / are you aware of stack overflow careers 2.0?,what type of project are you developing?,extraversion,please rate the advertising you've seen on stack overflow (the ads are relevant),frequency_1,stackoverflowanswer,sofriction,important_buildexisting,excoderbelonged,aiagentchallengesstrongly disagree,importanthiringcompanies,toolstechadmired,jobsatpoints_8,how did you find out about your current job?,knowledge_8,updatecv,lastnewjob,purchasehow,visit_frequency,age_midpoint,haveworkedplatform,adsagreedisagree2,commplatformadmired,do you have a stack overflow careers profile?,toolcountwork,so_actions_15,"what is your budget for outside expenditures (hardware, software, consulting, etc) for 2011?",aiacc,haveworkedframework,assessbenefits2,aiagentobservesecure,stackoverflowhasaccount,newlearn,select up to 3 (how can companies improve interview process: more live code),devenvschoice,jobsatpoints_2,seriouswork,languagedesirenextyear,competepeers,assessjobcommute,select all that apply (future lang & tech: android),agree_tech,cousineducation,stackoverflowdescribes,"in an average week, how do you spend your time? (developing new features)",assessjoboffice,compfreq,officestacksyncadmired,aimodelsadmired,toolstechhaveworkedwith,newrole,coderev,blockchainorg,newjobhunt,importanthiringrep,annoyingui,webframeworkedwith,influencecommunication,learncodeonline,opsysprofessional use,open_to_new_job,aihuman,adspriorities1,learningnewtech,"what is your budget for outside expenditures (hardware, software, consulting, etc) for 2014?",important_wfh,timeafterbootcamp,jobsatpoints_10,assessjobprojects,languageswantentry,workplan,stackoverflowmetachat,student,knowledge_5,commit_frequency,mainbranch,how many developers are employed at your company?,jobsearchstatus,educationtypes,aidevhaveworkedwith,aiagent_uses,stackoverflowdevstory,importanthiringeducation,aiagentorchestration,hypotheticaltools2,jobsatpoints_9,stackoverflowjobs,have you changed jobs in the last 12 months?,toolcountpersonal,assessbenefits11,accessibility,airesponsible,projectmanagement,languagehaveworkedwith,experience_midpoint,agree_loveboss,languagechoice,webframedesirenextyear,commplatformhaveentr,commplatformwanttoworkwith,frustration,purchasewhat,techoppose_13,techendorse_5,select up to 3 (most annoying about job search: finding time),jobemailpriorities6,star_wars_vs_star_trek,educationimportant,hoursperweek,select up to 3 (appealing message traits: message is personalized),salary,assessjob1,were you aware of the apptivate contest?,important_companymission,mgrmoney,newpurchaseresearch,agree_adblocker,assessbenefits7,responseid,aiethics,adsactions,expectedsalary,platformwantentry,agree_problemsolving,aiagentknowwrite,purchaseinfluence,stackoverflowjoblisting,jobsatpoints_15,screenname,aiagentchallengessomewhat agree,so_actions_3,salarytype,frequency_2,aiagentexternal,checkincode,careersat,newjobhuntresearch,auditoryenvironment,jobsatpoints_16,equipmentsatisfiedstorage,newpurplelink,aicomplex,techendorse_13,agentusesgeneral,sopartfreq,nondevelopertype,inthezone,interview_likelihood,stackoverflowjobsearch,assessjobexp,occupation_group,which of our sites do you frequent most?,how do you use stack overflow? (read other people's questions to solve my problems),"what is your budget for outside expenditu

In [20]:
combined_consolidated_df = df_all.copy()

### Consolidate values for country

In [46]:
# Find columns containing 'country'
country_columns = [col for col in combined_consolidated_df.columns if 'country' in str(col).lower()]

print("Columns containing 'country':")
for col in country_columns:
    try:
        # Select the column(s). If multiple columns share the same name this returns a DataFrame.
        selected = df_use.loc[:, col]
        # If a DataFrame is returned (duplicate column names), collapse to a single Series by taking
        # the first non-null value across duplicates for each row.
        if isinstance(selected, pd.DataFrame):
            if selected.shape[1] > 1:
                print(f"\nWarning: column name '{col}' is duplicated ({selected.shape[1]} columns). Combining duplicates by taking first non-null value.")
            series = selected.bfill(axis=1).iloc[:, 0]
        else:
            series = selected.squeeze()

        print(f"\nColumn: {col}")
        print("Top 5 most common values and their counts:")
        counts = series.fillna('NULL').value_counts().head()
        print(counts)

        # Get unique count excluding nulls
        unique_count = series.dropna().nunique()
        print(f"\nTotal unique values (excluding nulls): {unique_count}")
        print(f"Number of null values: {series.isnull().sum()}")
        print("-" * 50)
    except Exception as e:
        print(f"\nError processing column {col}: {str(e)}")
        print("-" * 50)


Columns containing 'country':


Column: country
Top 5 most common values and their counts:
country
United States               78721
NULL                        76931
India                       70128
United States of America    65806
Germany                     50561
Name: count, dtype: int64

Total unique values (excluding nulls): 271
Number of null values: 76931
--------------------------------------------------


In [22]:
# Mapping of variants -> canonical names (lowercased keys for matching)
country_map = {
    'united states': 'United States',
    'united states of america': 'United States',
    'united kingdom of great britain and northern ireland': 'United Kingdom',
    'united kingdom': 'United Kingdom',
    'trinidad and tobago': 'Trinidad and Tobago',
    'trinidad & tobago': 'Trinidad and Tobago',
    'syrian arab republic': 'Syria',
    'syria': 'Syria',
    'other country (not listed above)': 'Other',
    'other (please specify)': 'Other',
    'myanmar, {burma}': 'Myanmar',
    'myanmar': 'Myanmar',
    'libyan arab jamahiriya': 'Libya',
    'libya': 'Libya',
    'laos': 'Laos',
    "lao people's democratic republic": 'Laos',
    'korea south': 'South Korea',
    'republic of korea': 'South Korea',
    'south korea': 'South Korea',
    'korea north': 'North Korea',
    'north korea': 'North Korea',
    'ireland': 'Ireland',
    'ireland {republic}': 'Ireland',
    'hong kong (s.a.r.)': 'Hong Kong',
    'hong kong': 'Hong Kong',
    'guinea-bissau': 'Guinea',
    'guinea': 'Guinea',
    'bosnia herzegovina': 'Bosnia and Herzegovina',
    'bosnia and herzegovina': 'Bosnia and Herzegovina',
    'bosnia-herzegovina': 'Bosnia and Herzegovina',
    'vatican city state': 'Vatican',
    'vatican': 'Vatican',
    'viet nam': 'Vietnam',
    'vietnam': 'Vietnam'
}

In [47]:
# Standardize and map 'country' values in combined_consolidated_df
def standardize_country(val):
    # Preserve NaN/None as-is
    if pd.isna(val):
        return val
    # Normalize to lower-case stripped string for lookup
    key = str(val).strip().lower()
    # Return mapped canonical name if available, otherwise return original (preserve original casing)
    return country_map.get(key, val)

# Apply mapping (overwrites 'country' column if present)
if 'country' in combined_consolidated_df.columns:
    combined_consolidated_df['country'] = combined_consolidated_df['country'].apply(standardize_country)
    print("Mapped 'country' values using country_map. Sample counts:")
    # Print the top values (including NaN) to give a quick check
    print(combined_consolidated_df['country'].value_counts(dropna=False).head(20))
else:
    print("Warning: 'country' column not found in combined_consolidated_df")

Mapped 'country' values using country_map. Sample counts:
country
United States         152658
India                  72378
Germany                51978
NaN                    50491
United Kingdom         46864
Canada                 26954
France                 22340
Poland                 17108
Brazil                 16764
Netherlands            15657
Australia              15134
Italy                  13758
Spain                  13201
Russian Federation     12869
Sweden                 10851
Ukraine                 9785
Switzerland             8294
Turkey                  7402
Israel                  7269
Austria                 7066
Name: count, dtype: int64


### Remove variables with 70% or more empty

In [48]:
null_pct = combined_consolidated_df.isna().mean()

In [49]:
# Remove columns from combined_consolidated_df with >70% null values
threshold = 0.7
cols_to_drop = null_pct[null_pct > threshold].index.tolist()
print(f"Dropping {len(cols_to_drop)} columns with >70% null values:")
print(cols_to_drop)
combined_consolidated_df.drop(columns=cols_to_drop, inplace=True)
print(f"New shape of combined_consolidated_df: {combined_consolidated_df.shape}")

Dropping 0 columns with >70% null values:
[]
New shape of combined_consolidated_df: (772667, 38)


In [50]:
# Detailed column stats sorted by non-null count (descending)
col_stats = pd.DataFrame({
    'non_null_count': combined_consolidated_df.notna().sum(),
    'null_count': combined_consolidated_df.isna().sum(),
    'unique_count': combined_consolidated_df.nunique(dropna=True)
}).sort_values('non_null_count', ascending=False)


In [51]:
# Add percent of total rows for nulls (rounded to 2 decimals)
total_rows = combined_consolidated_df.shape[0]
col_stats['null_pct'] = (col_stats['null_count'] / total_rows * 100).round(2)

In [52]:
col_stats

,non_null_count,null_count,unique_count,null_pct
gender,772667,0,7,0.00
surveyyear,772667,0,15,0.00
country,722176,50491,265,6.53
employment,704030,68637,228,8.88
education,686474,86193,12,11.16
years_coding,637461,135206,78,17.50
age,624959,147708,94,19.12
programming_experience,582775,189892,141,24.58
developer_type,566711,205956,18,26.66
company_size,541577,231090,38,29.91


### Consolidate values for 'gender'

In [53]:
combined_consolidated_df['gender'].value_counts()

gender
man                  414521
unknown              321117
woman                 29258
prefer_not_to_say      3466
non-binary             2845
other                  1233
transgender             227
Name: count, dtype: int64

In [54]:
# Keep the portion before the first semicolon in df_final['gender'] and convert to lower case
def keep_before_semicolon(val):
    if pd.isna(val):
        return val
    if isinstance(val, str):
        return val.split(';', 1)[0].strip()
    return val

combined_consolidated_df['gender_raw'] = combined_consolidated_df['gender'].apply(keep_before_semicolon)
combined_consolidated_df['gender_raw'] = combined_consolidated_df['gender_raw'].str.lower()

# Quick check
print(combined_consolidated_df['gender_raw'].value_counts(dropna=False).head(20))

gender_raw
man                  414521
unknown              321117
woman                 29258
prefer_not_to_say      3466
non-binary             2845
other                  1233
transgender             227
Name: count, dtype: int64


In [55]:
# Exact mapping for common raw values (keys are lowercased/stripped)
exact_map = {
    'nan': None,  # placeholder, handled below
    'man': 'man',
    'male': 'man',
    'woman': 'woman',
    'female': 'woman',
    'non-binary, genderqueer, or gender non-conforming': 'non-binary',
    'gender non-conforming': 'non-binary',
    'transgender': 'transgender',
    'prefer not to say': 'prefer_not_to_say',
    'prefer not to disclose': 'prefer_not_to_say',
    'or, in your own words:': 'other',
    'other': 'other'
}

def apply_exact_map(val):
    # Preserve NaN as unknown
    if pd.isna(val):
        return 'unknown'
    key = str(val).strip().lower()
    return exact_map.get(key, None)  # None if no exact mapping

# Create exact mapping column
combined_consolidated_df['gender_update'] = combined_consolidated_df['gender_raw'].apply(apply_exact_map)

# Quick summary counts
print("Exact-mapped counts:")
print(combined_consolidated_df['gender_update'].value_counts(dropna=False))

Exact-mapped counts:
gender_update
man            414521
None           327428
woman           29258
other            1233
transgender       227
Name: count, dtype: int64


In [56]:
# extract surveyyear and country and make a counts matrix
cols = ['surveyyear', 'gender_update']
df_sub = combined_consolidated_df[cols].copy()

# drop rows missing either value
#df_sub = df_sub.dropna(subset=['surveyyear', 'country'])

# ensure surveyyear is treated consistently (optional)
# df_sub['surveyyear'] = df_sub['surveyyear'].astype(str)

# Create matrix: rows = surveyyear, cols = country, values = counts
matrix_df = pd.crosstab(df_sub['surveyyear'], df_sub['gender_update']).sort_index()

# display and optionally save
display(matrix_df)
matrix_df.to_csv('surveyyear_by_gender_mapped_final_matrix.csv')

gender_update,man,other,transgender,woman
surveyyear,,,,
2014,6864,0,0,352
2016,51388,274,0,3202
2017,31890,225,71,2697
2018,59620,0,156,4409
2019,78100,0,0,6709
2020,46134,0,0,4038
2021,75428,413,0,4292
2022,65097,321,0,3559


In [57]:
# Clean up gender columns: drop old versions and rename gender_update to gender
columns_to_drop = ['gender', 'gender_raw', 'gender_mapped_exact']
existing_cols = [col for col in columns_to_drop if col in combined_consolidated_df.columns]

if existing_cols:
    combined_consolidated_df = combined_consolidated_df.drop(columns=existing_cols)
    print(f"Dropped columns: {existing_cols}")

if 'gender_update' in combined_consolidated_df.columns:
    combined_consolidated_df = combined_consolidated_df.rename(columns={'gender_update': 'gender'})
    print("Renamed 'gender_update' to 'gender'")

print(f"Final shape: {combined_consolidated_df.shape}")

Dropped columns: ['gender', 'gender_raw']
Renamed 'gender_update' to 'gender'
Final shape: (772667, 38)


### Consolidate years_coding

In [58]:
def clean_years_coding(val):
    """
    Clean years_coding values:
    - Convert ranges (e.g., '9-11 years') to median
    - Extract single numeric values
    - Drop values >100
    - Return NaN for non-numeric values
    """
    if pd.isna(val):
        return None
    
    val_str = str(val).strip().lower()
    
    # Extract numbers from ranges (e.g., "9-11 years" or "9-11")
    range_match = re.search(r'(\d+)\s*[-–—to]\s*(\d+)', val_str)
    if range_match:
        start = float(range_match.group(1))
        end = float(range_match.group(2))
        median_val = (start + end) / 2
        return median_val if median_val <= 100 else None
    
    # Extract single number (e.g., "5 years" or "5")
    single_match = re.search(r'(\d+(?:\.\d+)?)', val_str)
    if single_match:
        num_val = float(single_match.group(1))
        return num_val if num_val <= 100 else None
    
    # No numeric value found
    return None

# Apply cleaning function
combined_consolidated_df['years_coding_clean'] = combined_consolidated_df['years_coding'].apply(clean_years_coding)

# Show comparison
print(f"Min: {combined_consolidated_df['years_coding_clean'].min()}")
print(f"Max: {combined_consolidated_df['years_coding_clean'].max()}")
print(f"Mean: {combined_consolidated_df['years_coding_clean'].mean():.2f}")
print(f"Median: {combined_consolidated_df['years_coding_clean'].median()}")

Min: 1.0
Max: 100.0
Mean: 11.89
Median: 9.0


In [59]:
# Drop the old years_coding column and rename years_coding_clean
combined_consolidated_df = combined_consolidated_df.drop(columns=['years_coding'])
combined_consolidated_df = combined_consolidated_df.rename(columns={'years_coding_clean': 'years_coding'})

print(f"Updated columns. New shape: {combined_consolidated_df.shape}")
print(f"\nYears coding column summary:")
print(f"  Non-null count: {combined_consolidated_df['years_coding'].notna().sum()}")
print(f"  Min: {combined_consolidated_df['years_coding'].min()}")
print(f"  Max: {combined_consolidated_df['years_coding'].max()}")
print(f"  Mean: {combined_consolidated_df['years_coding'].mean():.2f}")

Updated columns. New shape: (772667, 38)

Years coding column summary:
  Non-null count: 637461
  Min: 1.0
  Max: 100.0
  Mean: 11.89


### Consolidate Education

In [60]:
def standardize_education(val):
    """
    Standardize education values:
    - bachelor's -> bachelors
    - master's -> masters
    - on-the-job training variations -> on_the_job_training
    """
    if pd.isna(val):
        return val
    
    val_str = str(val).strip().lower()
    
    # Check for bachelor's variations
    if "bachelor" in val_str or "b.a" in val_str or "b.s." in val_str or "b.s" in val_str:
        return "bachelors"
    
    # Check for master's variations
    if "master" in val_str:
        return "masters"
    
    # Check for on-the-job training variations
    if "self-taught" in val_str or "online class" in val_str or "self taught" in val_str:
        return "self-taught"

    # Check for on-the-job training variations
    if "on-the-job" in val_str or "on the job" in val_str or "job training" in val_str:
        return "on the job training"

    # Check for on-the-job training variations
    if "some college" in val_str:
        return "some college"
    
    # Check for on-the-job training variations
    if "primary" in val_str or "secondary" in val_str:
        return "less than college"

    # Check for on-the-job training variations
    if "associate" in val_str:
        return "associate degree"
    
    # Check for on-the-job training variations
    if "md" in val_str or "doctor" in val_str or "phd" in val_str or "doctoral" in val_str or "jd" in val_str:
        return "doctoral degree"
    
    # Check for on-the-job training variations
    if "full-time, intensive" in val_str or "part-time program" in val_str or "industry certification" in val_str or "mentorship program" in val_str or "something else" in val_str or "other" in val_str:
        return "other program"
    
    # Check for on-the-job training variations
    if "i prefer not to say" in val_str or "i prefer not to answer" in val_str:
        return "i prefer not to say"
    
    # Check for on-the-job training variations
    if "i never completed any formal education" in val_str:
        return "none"
    
    # Return original value if no match
    return val

# Apply standardization to education column
combined_consolidated_df['education_clean'] = combined_consolidated_df['education'].apply(standardize_education)

# Create a pivot-style DataFrame from education value counts
education_counts = combined_consolidated_df['education_clean'].value_counts().reset_index()
education_counts.columns = ['Education Level', 'Count']

# Add percentage column
total = education_counts['Count'].sum()
education_counts['Percentage'] = (education_counts['Count'] / total * 100).round(2)

# Display the formatted table
display(education_counts)

,Education Level,Count,Percentage
0,bachelors,305390,44.49
1,masters,152275,22.18
2,less than college,77015,11.22
3,some college,75726,11.03
4,doctoral degree,27906,4.07
5,self-taught,18490,2.69
6,associate degree,18380,2.68
7,other program,6364,0.93
8,none,2172,0.32
9,i prefer not to say,1109,0.16


In [61]:
# Drop the old education column and rename education_clean to education
combined_consolidated_df = combined_consolidated_df.drop(columns=['education'])
combined_consolidated_df = combined_consolidated_df.rename(columns={'education_clean': 'education'})

print(f"Updated columns. New shape: {combined_consolidated_df.shape}")
print(f"\nEducation column summary:")
print(f"  Non-null count: {combined_consolidated_df['education'].notna().sum()}")
print(f"\nValue counts:")
print(combined_consolidated_df['education'].value_counts())

Updated columns. New shape: (772667, 38)

Education column summary:
  Non-null count: 686474

Value counts:
education
bachelors              305390
masters                152275
less than college       77015
some college            75726
doctoral degree         27906
self-taught             18490
associate degree        18380
other program            6364
none                     2172
i prefer not to say      1109
on the job training       932
Professional degree       715
Name: count, dtype: int64


### Consolidate age

In [62]:
combined_consolidated_df['age'].value_counts()

age
29.5     166116
21.0      98493
39.5      89363
49.5      34924
27.0      29833
22.0      24258
32.0      19760
17.0      18706
59.5      13349
37.0      11124
20.0       8956
25.0       7362
28.0       6802
26.0       6793
24.0       6710
30.0       6457
23.0       6327
29.0       6211
44.5       4905
31.0       4783
33.0       4242
65.0       4130
35.0       3874
34.0       3768
45.0       3428
36.0       3209
38.0       2843
19.0       2346
40.0       2302
39.0       2265
18.0       1804
42.0       1650
54.5       1640
41.0       1567
43.0       1356
44.0       1096
16.0        920
46.0        888
48.0        860
47.0        805
60.0        770
50.0        755
49.0        722
15.0        645
55.5        540
52.0        531
51.0        496
53.0        455
55.0        425
54.0        408
14.0        357
56.0        337
57.0        291
58.0        273
59.0        225
13.0        202
62.0        179
61.0        164
63.0        148
12.0        145
64.0        111
66.0         81
67.0

In [63]:
def clean_age(val):
    """
    Clean age values:
    - Convert ranges (e.g., '25-34 years old') to median
    - Extract single numeric values
    - Handle special cases like 'Under 18' or '65 or older'
    - Force any values under 12 to be 12
    - Return NaN for non-numeric values
    """
    if pd.isna(val):
        return None
    
    val_str = str(val).strip().lower()
    result = None
    
    # Handle special cases
    if 'under' in val_str or 'less than' in val_str:
        # Extract the number after 'under' or 'less than'
        match = re.search(r'(\d+)', val_str)
        if match:
            result = float(match.group(1)) - 1  # e.g., "Under 18" -> 17
    
    elif 'older' in val_str or 'over' in val_str or 'above' in val_str:
        # Extract the number before 'older', 'over', or 'above'
        match = re.search(r'(\d+)', val_str)
        if match:
            result = float(match.group(1))  # e.g., "65 or older" -> 65
    
    # Extract numbers from ranges (e.g., "25-34" or "25 - 34")
    elif (range_match := re.search(r'(\d+)\s*[-–—to]\s*(\d+)', val_str)):
        start = float(range_match.group(1))
        end = float(range_match.group(2))
        result = (start + end) / 2
    
    # Extract single number (e.g., "25 years old" or "25")
    elif (single_match := re.search(r'(\d+)', val_str)):
        result = float(single_match.group(1))
    
    # Force minimum age of 12
    if result is not None and result < 12:
        result = 12
    
    return result

# Apply cleaning function to combined_consolidated_df
combined_consolidated_df['age_clean'] = combined_consolidated_df['age'].apply(clean_age)

# Show statistics
print(f"Age Statistics:")
print(f"Min: {combined_consolidated_df['age_clean'].min()}")
print(f"Max: {combined_consolidated_df['age_clean'].max()}")
print(f"Mean: {combined_consolidated_df['age_clean'].mean():.2f}")
print(f"Median: {combined_consolidated_df['age_clean'].median()}")
print(f"\nValue counts:")
print(combined_consolidated_df['age_clean'].value_counts().sort_index().head(50))

Age Statistics:
Min: 12.0
Max: 279.0
Mean: 31.28
Median: 29.0

Value counts:
age_clean
12.0       145
13.0       202
14.0       357
15.0       645
16.0       920
17.0     18706
18.0      1804
19.0      2346
20.0      8956
21.0     98493
22.0     24258
23.0      6327
24.0      6710
25.0      7362
26.0      6793
27.0     29833
28.0      6802
29.0    172327
30.0      6457
31.0      4783
32.0     19760
33.0      4242
34.0      3768
35.0      3874
36.0      3209
37.0     11124
38.0      2843
39.0     91628
40.0      2302
41.0      1567
42.0      1650
43.0      1356
44.0      6001
45.0      3428
46.0       888
47.0       805
48.0       860
49.0     35646
50.0       755
51.0       496
52.0       531
53.0       455
54.0      2048
55.0       965
56.0       337
57.0       291
58.0       273
59.0     13574
60.0       770
61.0       164
Name: count, dtype: int64


In [64]:
# Drop the old age column and rename age_clean to age
combined_consolidated_df = combined_consolidated_df.drop(columns=['age'])
combined_consolidated_df = combined_consolidated_df.rename(columns={'age_clean': 'age'})

print(f"Updated columns. New shape: {combined_consolidated_df.shape}")
print(f"\nAge column summary:")
print(f"  Non-null count: {combined_consolidated_df['age'].notna().sum()}")
print(f"  Min: {combined_consolidated_df['age'].min()}")
print(f"  Max: {combined_consolidated_df['age'].max()}")
print(f"  Mean: {combined_consolidated_df['age'].mean():.2f}")
print(f"  Median: {combined_consolidated_df['age'].median()}")

Updated columns. New shape: (772667, 38)

Age column summary:
  Non-null count: 624959
  Min: 12.0
  Max: 279.0
  Mean: 31.28
  Median: 29.0


### Consolidate Developer Type

In [65]:
combined_consolidated_df['developer_type'].value_counts()

developer_type
back-end developer                 146171
full-stack developer               114988
other developer                    111834
front-end developer                 52697
data role                           33640
research role                       19475
student                             17604
data scientist                      16648
engineering role                    13534
other role                          12700
design                              11524
executive                            4935
product/project manager              3313
systems administrator                3107
cyber security                       1942
educator                             1503
marketing or sales professional       638
retired                               458
Name: count, dtype: int64

In [66]:
# Keep only the portion before the first semicolon in developer_type and standardize values
def clean_developer_type(val):
    """
    Extract the first developer type before semicolon and standardize common variations.
    E.g., 'Full-stack developer;Back-end developer' -> 'full-stack developer'
    E.g., 'Developer, full-stack' -> 'full-stack developer'
    """
    if pd.isna(val):
        return val
    if isinstance(val, str):
        # Take only the first value before semicolon
        first_type = val.split(';', 1)[0].strip().lower()
        
        # Standardize full-stack variations
        if 'full-stack' in first_type or 'full stack' in first_type:
            return 'full-stack developer'

        if 'front-end' in first_type or 'front end' in first_type:
            return 'front-end developer'

        if 'back-end' in first_type or 'back end' in first_type:
            return 'back-end developer'
        
        if 'mobile' in first_type or 'architect' in first_type or 'graphics' in first_type or 'devops' in first_type or 'web' in first_type or 'developer' in first_type:
            return 'other developer'

        if 'data scien' in first_type or 'machine learning' in first_type or 'ML' in first_type or 'DS' in first_type :
            return 'data scientist'

        if 'data' in first_type :
            return 'data role'

        if 'engineer' in first_type :
            return 'engineering role'
        
        if 'research' in first_type or 'scientist' in first_type:
            return 'research role'
        
        if 'systems administrator' in first_type or 'system administrator' in first_type:
            return 'systems administrator'
        
        if 'executive' in first_type :
            return 'executive'
        
        if 'design' in first_type :
            return 'design'
        
        if 'security' in first_type or 'blockchain' in first_type:
            return 'cyber security'

        if 'product manager' in first_type or 'project manager' in first_type:
            return 'product/project manager'
        
        if 'other' in first_type :
            return 'other role'

        # Return the cleaned value
        return first_type
    return val

# Apply cleaning to df (which should be combined_consolidated_df based on context)
combined_consolidated_df['developer_type_clean'] = combined_consolidated_df['developer_type'].apply(clean_developer_type)

# Show value counts
print("Developer Type (cleaned) - Top 20:")
print(combined_consolidated_df['developer_type_clean'].value_counts().head(50))

Developer Type (cleaned) - Top 20:
developer_type_clean
back-end developer                 146171
full-stack developer               114988
other developer                    111834
front-end developer                 52697
data role                           33640
research role                       19475
student                             17604
data scientist                      16648
engineering role                    13534
other role                          12700
design                              11524
executive                            4935
product/project manager              3313
systems administrator                3107
cyber security                       1942
educator                             1503
marketing or sales professional       638
retired                               458
Name: count, dtype: int64


In [67]:
# Drop the old age column and rename age_clean to age
combined_consolidated_df = combined_consolidated_df.drop(columns=['developer_type'])
combined_consolidated_df = combined_consolidated_df.rename(columns={'developer_type_clean': 'developer_type'})

### Consolidate Remote Work Values

In [68]:
combined_consolidated_df['remote'].value_counts(dropna=False)

remote
NaN                                                                             389684
Hybrid (some remote, some in-person)                                             79167
Remote                                                                           62328
A few days each month                                                            32696
Less than once per month / Never                                                 30220
In-person                                                                        29115
Never                                                                            25527
Fully remote                                                                     25341
I rarely work remotely                                                           19212
All or almost all the time (I'm full-time remote)                                13370
Less than half the time, but at least one day each week                          10467
Full in-person                      

In [69]:
def standardize_remote(val):
    """
    Standardize remote work values into consistent categories:
    - fully_remote: Full-time remote workers
    - hybrid: Mix of remote and in-person
    - mostly_remote: Primarily remote with occasional in-person
    - mostly_in_person: Primarily in-person with occasional remote
    - fully_in_person: Full-time in-person
    - rarely_remote: Very infrequent remote work
    - never_remote: Never work remotely
    """
    if pd.isna(val):
        return val
    
    val_str = str(val).strip().lower()
    
    # Fully remote
    if any(term in val_str for term in [
        'fully remote',
        'full-time remote',
        'all or almost all the time',
        "i'm full-time remote"
    ]):
        return 'fully_remote'
    
    # Mostly remote (more than half the time)
    if any(term in val_str for term in [
        'more than half',
        'leans heavy to flexibility'
    ]):
        return 'mostly_remote'
    
    # Hybrid (balanced or unspecified)
    if any(term in val_str for term in [
        'hybrid (some remote, some in-person)',
        'about half the time',
        'your choice',
        'very flexible'
    ]):
        return 'hybrid'
    
    # Mostly in-person (some remote)
    if any(term in val_str for term in [
        'part-time remote',
        'less than half the time',
        'at least one day each week',
        'leans heavy to in-person',
        'i rarely work remotely',
        'a few days each month',
        'occasionally',
        'less than once per month'
    ]):
        return 'mostly in person'
    
    # Never remote
    if any(term in val_str for term in [
        'never',
        'full in-person',
        'fully in-person'
    ]):
        return 'fully_in_person'
    
    # Generic "remote" without specifics
    if val_str == 'remote':
        return 'fully_remote'
    
    # Generic "in-person" without specifics
    if val_str == 'in-person':
        return 'fully_in_person'
    
    # Catch-all for complicated/other cases
    if "complicated" in val_str:
        return 'hybrid'
    
    # Return original if no match
    return val

# Apply standardization
combined_consolidated_df['remote_clean'] = combined_consolidated_df['remote'].apply(standardize_remote)

# Show value counts
print("Remote Work Status (cleaned):")
print(combined_consolidated_df['remote_clean'].value_counts(dropna=False))
print(f"\nTotal non-null: {combined_consolidated_df['remote_clean'].notna().sum()}")
print(f"Total null: {combined_consolidated_df['remote_clean'].isna().sum()}")

Remote Work Status (cleaned):
remote_clean
NaN                 389684
mostly in person    110266
fully_remote        106673
hybrid               92690
fully_in_person      63238
mostly_remote        10116
Name: count, dtype: int64

Total non-null: 382983
Total null: 389684


In [70]:
# Drop the old age column and rename age_clean to age
combined_consolidated_df = combined_consolidated_df.drop(columns=['remote'])
combined_consolidated_df = combined_consolidated_df.rename(columns={'remote_clean': 'remote'})

### Export df_final

In [71]:
df_final = combined_consolidated_df.copy()

In [ ]:
df_final.to_csv('df_final.csv') 

In [ ]:
df = pd.read_csv('df_final.csv')

In [ ]:
df_final.head()

,Unnamed: 0,databasewanttoworkwith,trans,sexuality,soaccount,toolstechhaveworkedwith,respondent,responseid,languagewanttoworkwith,newsosites,socomm,languagehaveworkedwith,platformhaveworkedwith,webframehaveworkedwith,mainbranch,newcollabtoolshaveworkedwith,sovisitfreq,databasehaveworkedwith,newcollabtoolswanttoworkwith,ethnicity,sopartfreq,surveyyear,opsyspersonal use,employment,company_size,developer_type,country,programming_experience,dev_environment,operating_sys,learning,remote,survey_easy,currency,hobby,gender,years_coding,education,age,developer_type_clean
0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2011,NaN,NaN,Start Up (1-25),NaN,United States,NaN,NaN,Linux,NaN,NaN,NaN,NaN,NaN,unknown,11.0,NaN,32.0,NaN
1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2011,NaN,NaN,Start Up (1-25),NaN,Other Europe,NaN,NaN,Mac OS X,NaN,NaN,NaN,NaN,NaN,unknown,11.0,NaN,45.0,NaN
2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2011,NaN,NaN,Mature Small Business (25-100),NaN,South America,NaN,NaN,Windows 7,NaN,NaN,NaN,NaN,NaN,unknown,11.0,NaN,32.0,NaN
3,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2011,NaN,NaN,Mid Sized (100-999),NaN,Other Asia,NaN,NaN,Mac OS X,NaN,NaN,NaN,NaN,NaN,unknown,NaN,NaN,32.0,NaN
4,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2011,NaN,NaN,"Other (not working, consultant, etc.)",NaN,Other Europe,NaN,NaN,Linux,NaN,NaN,NaN,NaN,NaN,unknown,NaN,NaN,27.0,NaN
